In [1]:
import pandas as pd
processed_books = pd.read_csv(r"C:\Users\LOQ\project_books\.venv\datas\processed_books_with_bestseller_award.csv")

In [2]:
processed_books.loc[
    processed_books["title"].str.contains(
        "kokoro", case=False, na=False
    ),
    ["title", "author_clean"]
].head(20)

,title,author_clean
3,Kokoro,lafcadio hearn
4668,Kokoro,bishop of hippo soseki
4669,Kokoro,natsume sôseki
4670,Kokoro,sōseki natsume


In [3]:
# Step 1: don't fillna — track missingness
has_desc = processed_books['clean_description'].notna() & (processed_books['description'].str.strip().str.len() > 6)
has_cat  = processed_books['category_clean'].notna()  & (processed_books['categories'].str.strip().str.len()  > 0)

# Step 2: encode only real values; use a zero vector for missing
import numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

def encode_masked(series, mask):
    emb = np.zeros((len(series), 384), dtype=np.float32)
    if mask.any():
        emb[mask.values] = model.encode(
            series[mask].tolist(),
            normalize_embeddings=True,
            batch_size=64,
            show_progress_bar=True
        )
    return emb

title_embeddings = encode_masked(processed_books['title'].fillna(''), processed_books['title'].notna())
category_embeddings   = encode_masked(processed_books['categories'].fillna(''), has_cat)
description_embeddings  = encode_masked(processed_books['description'].fillna(''), has_desc)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Batches:   0%|          | 0/72 [00:00<?, ?it/s]

Batches:   0%|          | 0/66 [00:00<?, ?it/s]

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
title_similarity = cosine_similarity(title_embeddings)

category_similarity = cosine_similarity(category_embeddings)

description_similarity = cosine_similarity(description_embeddings)

In [7]:
def making_similarity_matrix(title_weight, category_weight, description_weight):
    
    final_similarity = (title_weight * title_similarity +
    category_weight * category_similarity +
    description_weight * description_similarity)

    return final_similarity

In [4]:
from rapidfuzz import fuzz

def authors_match(selected_authors, candidate_authors):
    for selected_author in selected_authors:
        for candidate_author in candidate_authors:

            score = fuzz.token_set_ratio(
                selected_author,
                candidate_author
            )

            if score >= 80:
                return True

    return False

In [5]:
def is_same_edition(book_idx, selected_idx):
    selected_title = processed_books.loc[
        selected_idx, "title_clean"]
    
    candidate_title = processed_books.loc[
        book_idx, "title_clean"]
    
    selected_authors = processed_books.loc[
        selected_idx, "author_set"]
    
    candidate_authors = processed_books.loc[
        book_idx, "author_set"]

    # Check authors
    author_score = authors_match(selected_authors, candidate_authors)
    # Check titles
    title_score = fuzz.token_set_ratio(selected_title, candidate_title)

    if title_score >= 80 and author_score == True:
        return True

In [6]:
indices = pd.Series(processed_books.index, index=processed_books["title_clean"]).drop_duplicates()

In [7]:
import re
import unicodedata
def clean_title(title):
    title = unicodedata.normalize("NFKD", title)     # Remove accents
    title = title.encode("ascii", "ignore").decode("ascii")
    title = str(title).lower() 
    title = re.sub(r"\([^)]*\)", "", title)     # Remove text in parentheses
    title = re.sub(r"[^a-z0-9\s]", " ", title)     # Remove punctuation
    title = re.sub(r"\s+", " ", title).strip()     # Normalize spaces
    return title

In [8]:
def recommend_books(title, n, similarity_matrix, author_bonus):
    title_clean = clean_title(title)

    if title_clean not in indices:
        print("Book not found.")
        return pd.DataFrame()

    idc = indices[title_clean]

    if isinstance(idc, pd.Series):
        candidate_indices = [int(x) for x in idc.values]
    else:
        candidate_indices = [int(idc)]
    idx = candidate_indices[0]

    target_authors = processed_books.loc[idx, "author_set"]

    similarity_scores = sorted(
        enumerate(similarity_matrix[idx]),
        key=lambda x: x[1],
        reverse=True
    )[:100]

    AWARD_BONUS = 0.02
    BESTSELLER_BONUS = 0.02

    recommendations = []

    for book_idx, similarity in similarity_scores:
        if book_idx == idx:
            continue

        same_edition = is_same_edition(idx, book_idx)
        if same_edition:
            continue

        candidate_book = processed_books.iloc[book_idx]
        candidate_authors = candidate_book["author_set"]
        same_author = authors_match(target_authors, candidate_authors)
        current_author_bonus = author_bonus if same_author else 0.0
        same_edition = is_same_edition(idx, book_idx)

        # Apply award and bestseller bonuses independently
        is_award_winner = (
            candidate_book.get("is_award_winner", False) is True
            or str(candidate_book.get("is_award_winner", "")).strip().lower()
            in ("true", "1", "yes")
        )

        is_bestseller = (
            candidate_book.get("is_bestseller", False) is True
            or str(candidate_book.get("is_bestseller", "")).strip().lower()
            in ("true", "1", "yes")
        )
        award_bonus = AWARD_BONUS if is_award_winner else 0.0
        bestseller_bonus = BESTSELLER_BONUS if is_bestseller else 0.0

        final_score = (
            similarity
            + award_bonus
            + bestseller_bonus
            + current_author_bonus
        )

        recommendations.append({
            "index": book_idx,
            "similarity": similarity,
            "author_bonus": current_author_bonus,
            "award_bonus": award_bonus,
            "bestseller_bonus": bestseller_bonus,
            "final_score": final_score,
            "same_book": same_edition,
            "is_award_winner": is_award_winner,
            "is_bestseller": is_bestseller
        })

    recommendations.sort(
        key=lambda x: x["final_score"],
        reverse=True
    )

    recommendations = recommendations[:n]

    if not recommendations:
        return pd.DataFrame()

    book_indices = [x["index"] for x in recommendations]

    results = processed_books.iloc[book_indices][
        ["title", "authors", "categories"]
    ].copy()

    results["similarity"] = [
        x["similarity"] for x in recommendations
    ]
    results["author_bonus"] = [
        x["author_bonus"] for x in recommendations
    ]
    results["award_bonus"] = [
        x["award_bonus"] for x in recommendations
    ]
    results["bestseller_bonus"] = [
        x["bestseller_bonus"] for x in recommendations
    ]
    results["final_score"] = [
        x["final_score"] for x in recommendations
    ]
    results["same_book"] = [
        x["same_book"] for x in recommendations
    ]
    results["is_award_winner"] = [
        x["is_award_winner"] for x in recommendations
    ]
    results["is_bestseller"] = [
        x["is_bestseller"] for x in recommendations
    ]

    return results


In [13]:
print(processed_books["title"].dropna().unique().tolist())

['The Face of Another', 'Generations of Winter', 'The Street of Crocodiles', 'Kokoro', 'Sense and Sensibility - Illustrated Edition', 'The Final Problem', 'A House-boat on the Styx', 'Virgin Soil', 'Hemingway', 'Hunger', 'Italian Science Fiction', 'Oblomov', 'Principles of Phisics, or Natural Philosophy', 'Consolations in Travel; or, the Last Days of a Philosopher', 'The Three Musketeers', 'The Pickwick Papers', 'The Sorrows of Satan Illustrated', '西游记', 'The Portrait of a Lady Illustrated', 'How I Found Livingstone', 'Peter Pan (Peter and Wendy) Illustrated', "A Child's Garden of Verses Annotated", 'The Three Musketeers Annotated', 'Meridiana', "Professor Martens' Departure", 'The Hand of Ethelberta Illustrated', 'And Then There Were None', 'The Dwarves of Death', 'Around the World in 80 Days Illustrated', 'Typee Illustrated', 'Death in the Afternoon', 'Rhys Lewis, Minister of Bethel', 'The Little Lady of the Big House Illustrated', 'Robur the Conqueror Illustrated', 'Inherent Vice', 

In [14]:
test_cases = {
    "The Hobbit": [
        "The Eye of the World",
        "The Dragonbone Chair",
        "The Neverending Story",
        "The Golden Compass",
        "The Chronicles of Narnia - the Horse and His Boy",
        "The Wind in the Willows",
        "The Princess Bride",
    ],

    "Kokoro": [
        "No Longer Human",
        "The Setting Sun",
        "Snow Country",
        "Norwegian Wood",
        "Kitchen",
        "The Stranger",
        "The Metamorphosis",
    ],

    "No Longer Human": [
        "Kokoro",
        "The Setting Sun",
        "Norwegian Wood",
        "Kitchen",
        "The Stranger",
        "The Metamorphosis",
        "Notes from Underground",
    ],

    "The Shadow of the Wind": [
        "The Name of the Rose",
        "The Thirteenth Tale",
        "The Athenian Murders",
        "The Girl with the Dragon Tattoo",
        "The Clockmaker's Daughter",
        "The Book Thief",
        "One Hundred Years of Solitude",
    ],

    "And Then There Were None": [
        "The ABC Murders",
        "Murder on the Orient Express",
        "Death on the Nile",
        "The Murder at the Vicarage",
        "A Murder is Announced",
        "The Mysterious Affair at Styles (100 Copy Collector's Edition)",
        "The Moving Finger",
    ],

    "Gone Girl": [
        "The Girl on the Train",
        "The Girl with the Dragon Tattoo",
        "The Silent Patient",
        "The Talented Mr. Ripley",
        "The Shadow of the Wind",
        "The Lincoln Lawyer",
        "Alex Cross",
    ],

    "The Da Vinci Code": [
        "Angels & Demons",
        "The Lost Symbol",
        "The Name of the Rose",
        "The Athenian Murders",
        "The Shadow of the Wind",
        "The Clockmaker's Daughter",
        "The Beekeeper's Apprentice ; Or, on the Segregation of the Queen",
    ],

    "Dracula": [
        "Frankenstein",
        "Carmilla",
        "The Picture of Dorian Gray",
        "Dr Jekyll and Mr Hyde",
        "The Call of Cthulhu",
        "The Dunwich Horror and Others",
        "The Exorcist",
    ],

    "Alice's Adventures in Wonderland": [
        "Through the Looking-Glass",
        "The Neverending Story",
        "The Wonderful Wizard of Oz",
        "Peter Pan",
        "Momo",
        "The Wind in the Willows",
        "Winnie-the-Pooh",
    ],

    "Matilda": [
        "Charlie and the Chocolate Factory",
        "James and the Giant Peach",
        "The BFG",
        "The Witches",
        "The Ickabog",
        "Archer's Goon",
        "The Borrowers",
    ],

    "Charlotte's Web": [
        "Stuart Little",
        "The Wind in the Willows",
        "The Tale of Peter Rabbit",
        "The Jungle Book",
        "Black Beauty",
        "Watership Down",
        "Winnie-the-Pooh",
    ],

    "The Little Prince": [
        "Momo",
        "The Neverending Story",
        "The Prophet",
        "Jonathan Livingston Seagull",
        "The Alchemist",
        "The Happy Prince and Other Stories",
        "Winnie-the-Pooh",
    ],

    "The Fault in Our Stars": [
        "Me Before You",
        "Love Story",
        "Five Feet Apart",
        "Looking for Alaska",
        "The Bridges of Madison County",
        "Norwegian Wood",
        "All Your Perfects",
    ],

    "It Ends with Us": [
        "All Your Perfects",
        "Me Before You",
        "The Bridges of Madison County",
        "The Fault in Our Stars",
        "The Women's Room",
        "The Thorn Birds",
        "The House Mate",
    ],

    "Night": [
        "The Diary of Anne Frank",
        "Man's Search for Meaning",
        "The Book Thief",
        "All the Light We Cannot See",
        "The Boy in the Striped Pyjamas",
        "Angela's Ashes",
        "Long Walk to Freedom",
    ],

    "Man's Search for Meaning": [
        "Night",
        "The Diary of Anne Frank",
        "Tuesdays with Morrie",
        "The Story of My Experiments with Truth",
        "Long Walk to Freedom",
        "The Road Less Traveled",
        "The 7 Habits of Highly Effective People",
    ],

    "A Brief History of Time": [
        "Cosmos",
        "The Elegant Universe",
        "The Fabric of the Cosmos",
        "Pale Blue Dot",
        "The Demon-Haunted World",
        "The Andromeda Nebula",
        "The Naked Ape",
    ],

    "How to Win Friends and Influence People": [
        "The 7 Habits of Highly Effective People",
        "The Power of Positive Thinking",
        "What Color Is Your Parachute?",
        "Your Erroneous Zones",
        "The Subtle Art of Not Giving a Fuck",
        "Who Moved My Cheese?",
        "The Secret",
    ],

    "Cosmos": [
        "A Brief History of Time",
        "Pale Blue Dot",
        "The Demon-Haunted World",
        "The Elegant Universe",
        "The Fabric of the Cosmos",
        "The Andromeda Nebula",
        "The Naked Ape",
    ],

    "The Count of Monte Cristo": [
        "The Three Musketeers Annotated",
        "The Prisoner of Zenda",
        "The Scarlet Pimpernel (Annotated)",
        "Treasure Island",
        "The Lost World",
        "Around the World in Eighty Days",
        "A Tale of Two Cities",
    ],

    "The ABC Murders": [
        "Murder on the Orient Express",
        "Death on the Nile",
        "The Murder at the Vicarage",
        "A Murder is Announced",
        "The Moving Finger",
        "The Hound of the Baskervilles",
        "The Complete Sherlock Holmes",
    ],
}


In [15]:
catalog_titles = set(processed_books["title"].dropna())
for title in test_cases:
    title = clean_title(title)
    
missing_queries = [
    title for title in test_cases
    if title not in catalog_titles
]

missing_relevant = {
    query: [
        title for title in relevant
        if title not in catalog_titles
    ]
    for query, relevant in test_cases.items()
    if any(title not in catalog_titles for title in relevant)
}

print("Number of test cases:", len(test_cases))
print("Missing query books:", missing_queries)
print("Missing relevant books:", missing_relevant)

Number of test cases: 21
Missing query books: ['How to Win Friends and Influence People']
Missing relevant books: {'Kokoro': ['The Setting Sun', 'Snow Country', 'The Stranger'], 'No Longer Human': ['The Setting Sun', 'The Stranger', 'Notes from Underground'], 'The Shadow of the Wind': ['The Name of the Rose', 'The Thirteenth Tale', 'The Book Thief'], 'Gone Girl': ['The Silent Patient', 'The Talented Mr. Ripley', 'Alex Cross'], 'The Da Vinci Code': ['The Name of the Rose'], "Alice's Adventures in Wonderland": ['Through the Looking-Glass'], 'Matilda': ['The BFG', 'The Witches'], 'The Fault in Our Stars': ['Me Before You', 'Love Story', 'Five Feet Apart', 'Looking for Alaska'], 'It Ends with Us': ['Me Before You'], 'Night': ['The Book Thief'], "Man's Search for Meaning": ['The Story of My Experiments with Truth', 'The Road Less Traveled'], 'A Brief History of Time': ['The Elegant Universe', 'The Fabric of the Cosmos', 'Pale Blue Dot', 'The Demon-Haunted World', 'The Andromeda Nebula'], 'H

In [16]:
dataset_titles = set(
    processed_books["title"].dropna().str.strip()
)
clean_test_cases = {}

for query, relevant_books in test_cases.items():
    clean_test_cases[query] = [
        book
        for book in relevant_books
        if book in dataset_titles and book != query
    ]
test_cases = clean_test_cases

print("Number of test cases:", len(test_cases))
print("\nRelevant titles remaining per query:")

for query, relevant_books in test_cases.items():
    print(f"{query}: {len(relevant_books)}")


Number of test cases: 21

Relevant titles remaining per query:
The Hobbit: 7
Kokoro: 4
No Longer Human: 4
The Shadow of the Wind: 4
And Then There Were None: 7
Gone Girl: 4
The Da Vinci Code: 6
Dracula: 7
Alice's Adventures in Wonderland: 6
Matilda: 5
Charlotte's Web: 7
The Little Prince: 7
The Fault in Our Stars: 3
It Ends with Us: 6
Night: 6
Man's Search for Meaning: 5
A Brief History of Time: 2
How to Win Friends and Influence People: 5
Cosmos: 2
The Count of Monte Cristo: 6
The ABC Murders: 7


In [17]:
catalog_titles = set(processed_books["title"])

missing_relevant = {}

for query, relevant in test_cases.items():
    missing = [book for book in relevant if book not in catalog_titles]

    if missing:
        missing_relevant[query] = missing

print("Missing relevant books:")
for query, books in missing_relevant.items():
    print(query, "->", books)

Missing relevant books:


In [18]:
def precision_at_k(recommendations, relevant_books, k=10):
    recommendations = recommendations.head(k)
    relevant_books = {
        str(book).lower().strip()
        for book in relevant_books
    }
    recommended_books = {
        str(book).lower().strip()
        for book in recommendations["title"]
    }
    matches = len(
        recommended_books.intersection(relevant_books)
    )

    return matches / k

In [19]:

def recall_at_k(recommendations, relevant_books, k=10):
        # Take only the top k recommendations
    recommendations = recommendations.head(k)
        # Normalize relevant book titles
    relevant_books = {
        str(book).lower().strip()
        for book in relevant_books
    }
        # Normalize recommended book titles
    recommended_books = {
        str(book).lower().strip()
        for book in recommendations["title"]
    }
        # Count relevant books found in the recommendations
    matches = len(
        recommended_books & relevant_books
    )
        # Avoid division by zero
    if not relevant_books:
        return 0.0
    
    return matches / len(relevant_books)


In [20]:
import pandas as pd

# Normalized weights
title_weights = [0.475, 0.50, 0.525, 0.55, 0.575, 0.60, 0.625]
category_weights = [0.00, 0.025, 0.05, 0.075, 0.10, 0.15]
author_bonuses = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

results = []

# Count valid weight combinations
valid_weight_combinations = []

for title_w in title_weights:
    for category_w in category_weights:
        description_weight = 1 - title_w - category_w

        if description_weight <= 0:
            continue

        valid_weight_combinations.append(
            (title_w, category_w, description_weight)
        )

total_combinations = (
    len(valid_weight_combinations) * len(author_bonuses)
)

combination_number = 0

for title_w, category_w, description_weight in valid_weight_combinations:
    for author_bonus in author_bonuses:
        combination_number += 1

        print(
            f"\rTesting {combination_number}/{total_combinations}",
            end=""
        )

        # Create similarity matrix
        similarity_matrix_done = making_similarity_matrix(
            title_w,
            category_w,
            description_weight
        )

        precision_scores = []
        recall_scores = []

        # Evaluate every test case
        for test_book, relevant_books in test_cases.items():
            recommendations = recommend_books(
                title=test_book,
                n=10,
                similarity_matrix=similarity_matrix_done,
                author_bonus=author_bonus
            )

            if recommendations is None or recommendations.empty:
                precision_scores.append(0.0)
                recall_scores.append(0.0)
                continue

            precision = precision_at_k(
                recommendations,
                relevant_books,
                k=10
            )

            recall = recall_at_k(
                recommendations,
                relevant_books,
                k=10
            )

            precision_scores.append(precision)
            recall_scores.append(recall)

        # Average scores across all test cases
        average_precision = sum(precision_scores) / len(precision_scores)
        average_recall = sum(recall_scores) / len(recall_scores)

        results.append({
            "title_weight": title_w,
            "category_weight": category_w,
            "description_weight": description_weight,
            "author_bonus": author_bonus,
            "precision@10": average_precision,
            "recall@10": average_recall,
            "tested_books": len(test_cases)
        })

print("\n\nGrid search complete.")

# Convert results to a DataFrame
results_df = pd.DataFrame(results)

# Calculate F1 score
denominator = (
    results_df["precision@10"] + results_df["recall@10"]
)

results_df["f1@10"] = (
    2 * results_df["precision@10"] * results_df["recall@10"]
    / denominator.where(denominator != 0)
).fillna(0)

# Sort by F1, then precision
results_df = results_df.sort_values(
    by=["f1@10", "precision@10"],
    ascending=False
).reset_index(drop=True)

print("\nTop 10 configurations:")
print(results_df.head(10).to_string(index=False))

# Select best configuration
best_params = results_df.iloc[0].to_dict()

print("BEST PARAMETERS")
print("Title weight:", best_params["title_weight"])
print("Category weight:", best_params["category_weight"])
print("Description weight:", best_params["description_weight"])
print("Author bonus:", best_params["author_bonus"])
print("Precision@10:", best_params["precision@10"])
print("Recall@10:", best_params["recall@10"])
print("F1@10:", best_params["f1@10"])

results_df.to_csv(
    "sbert_grid_search_results.csv",
    index=False
)

print("\nResults saved to sbert_grid_search_results.csv")


Testing 336/336

Grid search complete.

Top 10 configurations:
 title_weight  category_weight  description_weight  author_bonus  precision@10  recall@10  tested_books    f1@10
        0.475             0.05               0.475          0.20      0.061905        0.1            21 0.076471
        0.475             0.05               0.475          0.25      0.061905        0.1            21 0.076471
        0.475             0.05               0.475          0.30      0.061905        0.1            21 0.076471
        0.475             0.05               0.475          0.35      0.061905        0.1            21 0.076471
        0.475             0.05               0.475          0.40      0.061905        0.1            21 0.076471
        0.500             0.05               0.450          0.15      0.061905        0.1            21 0.076471
        0.500             0.05               0.450          0.20      0.061905        0.1            21 0.076471
        0.500             0.05   

In [23]:
BEST_TITLE_WEIGHT = 0.475
BEST_CATEGORY_WEIGHT = 0.05
BEST_DESCRIPTION_WEIGHT = 0.47500000000000003
BEST_AUTHOR_BONUS = 0.2

best_similarity_matrix = making_similarity_matrix(
    BEST_TITLE_WEIGHT,
    BEST_CATEGORY_WEIGHT,
    BEST_DESCRIPTION_WEIGHT
)

In [24]:
error_analysis = []
for test_book, relevant_books in test_cases.items():
    recommendations = recommend_books(
        test_book,
        n=10,
        author_bonus=BEST_AUTHOR_BONUS,
        similarity_matrix=best_similarity_matrix
    )
    if recommendations.empty:
        continue
    recommended_titles = (
        recommendations["title"]
        .tolist()
    )
    # Which relevant books were retrieved?
    found = [
        book
        for book in relevant_books
        if book in recommended_titles
    ]
    
    # Which relevant books were missed?
    missed = [
        book
        for book in relevant_books
        if book not in recommended_titles
    ]
    # Ranking of each relevant book
    rankings = {}

    for book in relevant_books:
        if book in recommended_titles:
            rank = (
                recommended_titles.index(book)
                + 1
            )
            rankings[book] = rank

        else:
            rankings[book] = None

    error_analysis.append({
        "query": test_book,
        "relevant_books": relevant_books,
        "found": found,
        "missed": missed,
        "recommended": recommended_titles,
        "rankings": rankings
    })
    error_analysis_df = pd.DataFrame(error_analysis)
    error_analysis_df.to_csv("error_analysis", index=False)